# Reproduce mmCIF D-residue survey (Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/Reproduce_mmCIF_D_Residue_Survey.ipynb)

Re-verify the **16 known D-residue error structures** using **native mmCIF** from RCSB (not legacy `.pdb`).

**What this closes:** the manuscript limitation that the primary survey used legacy PDB format and excluded mmCIF-only deposits. This notebook shows the same **29 D-label/L-coordinate mismatches** persist when coordinates are read from mmCIF.

**Runtime:** ~1–3 minutes on Colab (network download of ≤16 CIF files + gemmi parse).

**Not claimed:** a full re-survey of every mmCIF-only PDB entry (~245+ structures for the primary CCD codes). That remains future work; this notebook is the publication-facing closeout for known errors.


In [ ]:
# Cell 1 — Install gemmi + numpy
!pip -q install gemmi numpy pandas


In [ ]:
# Cell 2 — Clone repo for scripts + frozen results
import os, subprocess
if not os.path.isdir('benchmarks'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Tommaso-R-Marena/ChiralFold.git', 'ChiralFold'], check=True)
    os.chdir('ChiralFold')
print('cwd:', os.getcwd())


In [ ]:
# Cell 3 — Run mmCIF expansion (all 16 known-error structures)
# Equivalent CLI: python benchmarks/mmcif_d_residue_expansion.py
!python benchmarks/mmcif_d_residue_expansion.py


In [ ]:
# Cell 4 — Compare mmCIF results to frozen legacy-PDB survey
import json
from pathlib import Path
import pandas as pd

summary = json.loads(Path('results/mmcif_d_residue_expansion_summary.json').read_text())
legacy = json.loads(Path('results/d_residue_verification_summary.json').read_text())
df = pd.read_csv('results/mmcif_d_residue_expansion.csv')

print('mmCIF structures scanned:', summary['n_structures'])
print('mmCIF D-residues:', summary['n_d_residues'])
print('mmCIF errors (V>0):', summary['n_errors'])
print('Error PDBs:', ', '.join(summary['error_pdbs']))
print()
print('Legacy PDB survey errors:', legacy['l_error'])
print('Legacy structures with errors:', len(legacy['errors_by_structure']))

assert summary['n_errors'] == legacy['l_error'] == 29
assert set(summary['error_pdbs']) == set(legacy['errors_by_structure'])
print('\nPASS — mmCIF recovers the same 29 errors in the same 16 structures.')
display(df[df['is_error']].sort_values(['pdb_id', 'resnum']))


## How to interpret

| Question | Answer |
|----------|--------|
| Do the 29 errors disappear in mmCIF? | **No** — all 29 remain with V>0. |
| Were mmCIF-only deposits in the primary survey? | **No** — the numpy PDB-format survey excluded them (~245 for DPN/DAS/DAL alone). |
| Is a full mmCIF-universe audit done? | **Not in this notebook** — scoped re-verification of known errors for publication. |

CLI equivalent (laptop / Rockfish CPU node):

```bash
pip install gemmi numpy
python benchmarks/mmcif_d_residue_expansion.py
# optional smoke: python benchmarks/mmcif_d_residue_expansion.py --limit 5
```
